# Knowledge Loom → FORRT Nanopublications (Batch)

Process all non-DONE records from `loom_records.txt` in one go.

## How to run
1. Run all Setup cells (section 1)
2. Run **Generate** — creates TriG files for all records into `outputs/<ID>/`
3. Review the generated files
4. Run **Publish** — signs and publishes in correct order per record
5. Mark published records as DONE in `loom_records.txt`

For single-record processing with previews, use `loom2nanopub.ipynb`.


## 1. Setup

In [36]:
import json
import re
import urllib.parse
import urllib.request
from datetime import datetime, timezone
from dataclasses import dataclass
from pathlib import Path

from rdflib import Dataset, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, XSD, FOAF
from nanopub import load_profile

In [ ]:
# Configuration
PROFILE_PATH = "/Users/annef/Documents/ScienceLive/annefou-profile/profile.yml"
NP_CMD = "/Users/annef/Documents/ScienceLive/bin/np"
NP_SERVER = "https://np.knowledgepixels.com/"
OUTPUT_DIR = "../outputs/"

profile = load_profile(PROFILE_PATH)
AUTHOR_URI = URIRef(profile.orcid_id)  # CRITICAL: URIRef not ORCID[...]
AUTHOR_NAME = profile.name
print(f"Profile: {AUTHOR_NAME} ({AUTHOR_URI})")
assert 'orcid.org/orcid.org' not in str(AUTHOR_URI), "Double ORCID!"


In [38]:
# Namespaces
PUBLISHER = "https://sciencelive4all.org/"
TEMP_NP = Namespace("https://w3id.org/sciencelive/np")
NP = Namespace("http://www.nanopub.org/nschema#")
DCT = Namespace("http://purl.org/dc/terms/")
NT = Namespace("https://w3id.org/np/o/ntemplate/")
NPX = Namespace("http://purl.org/nanopub/x/")
PROV = Namespace("http://www.w3.org/ns/prov#")
SCHEMA = Namespace("http://schema.org/")
SCIENCELIVE = Namespace("https://w3id.org/sciencelive/o/terms/")

FORRT_CLAIM_TEMPLATE = URIRef("https://w3id.org/np/RAu5uTahAxc0OLBB3vaGwK3OQDDZV7QuWtDlBk0Ea3bco")
FORRT_KL_STUDY_TEMPLATE = URIRef("https://w3id.org/np/RALIq4JelUP-q9BuWONcKMJ87B5n59ppcwhQjl-1dheO4")
FORRT_KL_OUTCOME_TEMPLATE = URIRef("https://w3id.org/np/RAw3XdUhxQJfKBaU-cQhV6c7au4rLd5CSUdbMKTS_FB8g")
PROV_TEMPLATE = URIRef("https://w3id.org/np/RA7lSq6MuK_TIC6JMSHvLtee3lpLoZDOqLJCLXevnrPoU")
PUBINFO_TEMPLATE_1 = URIRef("https://w3id.org/np/RACJ58Gvyn91LqCKIO9zu1eijDQIeEff28iyDrJgjSJF8")
PUBINFO_TEMPLATE_2 = URIRef("https://w3id.org/np/RAukAcWHRDlkqxk7H2XNSegc1WnHI569INvNr-xdptDGI")

KL_API = "https://knowledgeloom.tib.eu/api/v1"
DTREG_TYPE_MAP = {
    "feeb33ad3e4440682a4d": SCIENCELIVE["dtreg-DataAnalysis"],
    "37182ecfb4474942e255": SCIENCELIVE["dtreg-DataPreprocessing"],
    "5b66cb584b974b186f37": SCIENCELIVE["dtreg-DescriptiveStatistics"],
    "b9335ce2c99ed87735a6": SCIENCELIVE["dtreg-GroupComparison"],
    "286991b26f02d58ee490": SCIENCELIVE["dtreg-RegressionAnalysis"],
    "3f64a93eef69d721518f": SCIENCELIVE["dtreg-CorrelationAnalysis"],
    "c6b413ba96ba477b5dca": SCIENCELIVE["dtreg-MultilevelAnalysis"],
    "6e3e29ce3ba5a0b9abfe": SCIENCELIVE["dtreg-ClassPrediction"],
    "c6e19df3b52ab8d855a9": SCIENCELIVE["dtreg-ClassDiscovery"],
    "5e782e67e70d0b2a022a": SCIENCELIVE["dtreg-AlgorithmEvaluation"],
    "437807f8d1a81b5138a3": SCIENCELIVE["dtreg-FactorAnalysis"]
}
print("Namespaces configured.")
# GitLab API (for dtreg JSON-LD files)
GITLAB_API = "https://gitlab.com/api/v4"
LOOM_GROUP = "TIBHannover/lki/knowledge-loom/loom-records"


Namespaces configured.


In [39]:
# Helpers
def fetch_json(url):
    with urllib.request.urlopen(urllib.request.Request(url)) as resp:
        return json.loads(resp.read().decode())

def gitlab_api_url(repo_path, endpoint):
    encoded = urllib.parse.quote(f"{LOOM_GROUP}/{repo_path}", safe="")
    return f"{GITLAB_API}/projects/{encoded}/{endpoint}"

def slugify(s):
    return re.sub(r'[^a-zA-Z0-9_-]', '-', s.lower()).strip('-')[:60]

def bind_all(ds):
    for p, n in [("this", TEMP_NP), ("sub", TEMP_NP), ("np", NP), ("dct", DCT),
                  ("nt", NT), ("npx", NPX), ("xsd", XSD), ("rdfs", RDFS),
                  ("prov", PROV), ("foaf", FOAF), ("schema", SCHEMA),
                  ("sciencelive", SCIENCELIVE)]:
        ds.bind(p, n)

def make_head(ds):
    this_np = URIRef(TEMP_NP)
    h = ds.graph(URIRef(TEMP_NP + "/Head"))
    h.add((this_np, RDF.type, NP.Nanopublication))
    h.add((this_np, NP.hasAssertion, URIRef(TEMP_NP + "/assertion")))
    h.add((this_np, NP.hasProvenance, URIRef(TEMP_NP + "/provenance")))
    h.add((this_np, NP.hasPublicationInfo, URIRef(TEMP_NP + "/pubinfo")))
    return this_np

def make_provenance(ds):
    p = ds.graph(URIRef(TEMP_NP + "/provenance"))
    p.add((URIRef(TEMP_NP + "/assertion"), PROV.wasAttributedTo, AUTHOR_URI))

def make_pubinfo(ds, this_np, label, template_uri, introduced_uri=None):
    pi = ds.graph(URIRef(TEMP_NP + "/pubinfo"))
    pi.add((AUTHOR_URI, FOAF.name, Literal(AUTHOR_NAME)))
    now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")
    pi.add((this_np, DCT.created, Literal(now, datatype=XSD.dateTime)))
    pi.add((this_np, DCT.creator, AUTHOR_URI))
    pi.add((this_np, DCT.license, URIRef("https://creativecommons.org/licenses/by/4.0/")))
    pi.add((this_np, NPX.wasCreatedAt, URIRef(PUBLISHER)))
    pi.add((this_np, RDFS.label, Literal(label)))
    pi.add((this_np, NT.wasCreatedFromTemplate, template_uri))
    pi.add((this_np, NT.wasCreatedFromProvenanceTemplate, PROV_TEMPLATE))
    pi.add((this_np, NT.wasCreatedFromPubinfoTemplate, PUBINFO_TEMPLATE_1))
    pi.add((this_np, NT.wasCreatedFromPubinfoTemplate, PUBINFO_TEMPLATE_2))
    if introduced_uri:
        pi.add((this_np, NPX.introduces, introduced_uri))

def sign_and_publish(trig_file, resource_suffix=None):
    """Sign and publish. Returns (nanopub_uri, resource_uri)."""
    from nanopub import Nanopub, NanopubConf
    conf = NanopubConf(profile=profile)
    np_obj = Nanopub(rdf=trig_file, conf=conf)
    np_obj.sign()
    
    # Store signed version for inspection
    signed_path = trig_file.with_suffix(".signed.trig")
    np_obj.store(signed_path)
    
    np_obj.publish()
    np_uri = np_obj.source_uri
    
    # Build the resource URI from the signed file
    resource_uri = None
    if resource_suffix:
        # The sub: namespace in signed file is np_uri + "/"
        # So resource is np_uri + "/" + suffix
        # But np_uri from library may be short form, get full from signed file
        content = signed_path.read_text()
        import re
        match = re.search(r"prefix sub\d*: <([^>]+)>", content)
        if match:
            resource_uri = match.group(1) + resource_suffix
    
    return np_uri, resource_uri

def validate_trig(trig_file):
    content = trig_file.read_text()
    assert 'orcid.org/https' not in content, f"Double ORCID in {trig_file.name}!"
    assert '10.82209/' not in content, f"Unresolvable TIB DOI in {trig_file.name}!"
    return True

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("Helpers defined.")

# Fetch dtreg JSON-LD files from GitLab for software/input data metadata
@dataclass
class AnalysisInfo:
    label: str = ""
    method_label: str = ""
    method_call: str = ""
    package_name: str = ""
    package_version: str = ""
    runtime_name: str = ""
    runtime_version: str = ""
    input_label: str = ""
    input_source_url: str = ""
    input_rows: int = 0
    input_cols: int = 0
    json_file: str = ""

def get_prop(obj, suffix):
    """Get a property from a dtreg JSON-LD object by suffix (after #)."""
    if not isinstance(obj, dict): return None
    for key, val in obj.items():
        if key.endswith(f"#{suffix}") or key == suffix: return val
    return None

def parse_single_step(step, filename):
    """Extract software/input metadata from one analysis step."""
    info = AnalysisInfo(json_file=filename)
    executes = get_prop(step, "executes")
    if isinstance(executes, dict):
        info.method_label = get_prop(executes, "label") or ""
        info.method_call = get_prop(executes, "is_implemented_by") or ""
        part_of = get_prop(executes, "part_of")
        if isinstance(part_of, dict):
            info.package_name = get_prop(part_of, "label") or ""
            info.package_version = get_prop(part_of, "version_info") or ""
            runtime = get_prop(part_of, "part_of")
            if isinstance(runtime, dict):
                info.runtime_name = get_prop(runtime, "label") or ""
                info.runtime_version = get_prop(runtime, "version_info") or ""
    has_input = get_prop(step, "has_input")
    # has_input can be a dict or list of dicts
    if isinstance(has_input, list):
        has_input = has_input[0] if has_input else None
    if isinstance(has_input, dict):
        info.input_label = get_prop(has_input, "label") or ""
        info.input_source_url = get_prop(has_input, "source_url") or ""
        chars = get_prop(has_input, "has_characteristic")
        if isinstance(chars, dict):
            info.input_rows = get_prop(chars, "number_of_rows") or 0
            info.input_cols = get_prop(chars, "number_of_columns") or 0
    method_str = f"{info.package_name}::{info.method_label}" if info.package_name else info.method_label
    info.label = method_str or filename
    return info if (info.method_label or info.input_label) else None

@dataclass
class StepOutputInfo:
    """Output data for a single analysis step (maps to one KL statement)."""
    step_label: str = ""
    target_variable: str = ""
    output_label: str = ""
    output_columns: list = None
    output_values: dict = None  # {col_name: value} for first row
    output_rows: int = 0
    output_cols: int = 0
    viz_url: str = ""
    input_label: str = ""
    input_source_url: str = ""

    def __post_init__(self):
        if self.output_columns is None: self.output_columns = []
        if self.output_values is None: self.output_values = {}

def extract_step_output(step):
    """Extract output/result data from a dtreg analysis step."""
    info = StepOutputInfo()
    info.step_label = get_prop(step, "label") or ""
    # Input data
    has_input = get_prop(step, "has_input")
    if isinstance(has_input, list): has_input = has_input[0] if has_input else None
    if isinstance(has_input, dict):
        info.input_label = get_prop(has_input, "label") or ""
        info.input_source_url = get_prop(has_input, "source_url") or ""
    targets = get_prop(step, "targets")
    if isinstance(targets, dict):
        info.target_variable = get_prop(targets, "label") or ""
    elif isinstance(targets, list) and targets:
        info.target_variable = ", ".join(get_prop(t, "label") or "" for t in targets if isinstance(t, dict))
    has_output = get_prop(step, "has_output")
    if isinstance(has_output, dict):
        info.output_label = get_prop(has_output, "label") or ""
        # Column names
        parts = get_prop(has_output, "has_part")
        if isinstance(parts, list):
            info.output_columns = [get_prop(p, "label") for p in parts if isinstance(p, dict)]
        # Dimensions
        chars = get_prop(has_output, "has_characteristic")
        if isinstance(chars, dict):
            info.output_rows = get_prop(chars, "number_of_rows") or 0
            info.output_cols = get_prop(chars, "number_of_columns") or 0
        # Visualization URL
        expr = get_prop(has_output, "has_expression")
        if isinstance(expr, dict):
            info.viz_url = get_prop(expr, "source_url") or ""
        # Actual result values from first row of source_table
        src_table = get_prop(has_output, "source_table")
        if isinstance(src_table, dict):
            cols = src_table.get("columns", get_prop(src_table, "columns") or [])
            rows = src_table.get("rows", get_prop(src_table, "rows") or [])
            col_titles = []
            for c in (cols if isinstance(cols, list) else []):
                title = c.get("col_titles", get_prop(c, "col_titles") or get_prop(c, "titles") or "")
                col_titles.append(title)
            if isinstance(rows, list) and rows:
                cells = rows[0].get("cells", get_prop(rows[0], "cells") or [])
                if isinstance(cells, list):
                    for ci, cell in enumerate(cells):
                        v = get_prop(cell, "value") or get_prop(cell, "primary_value") or ""
                        col_name = col_titles[ci] if ci < len(col_titles) else info.output_columns[ci] if ci < len(info.output_columns) else f"col{ci}"
                        info.output_values[col_name] = str(v)
    return info

def extract_analysis_steps(dtreg_data, filename):
    """Extract all analysis steps from a dtreg JSON-LD file. Returns list of AnalysisInfo."""
    results = []
    has_part = get_prop(dtreg_data, "has_part")
    # has_part can be a single dict or a list of steps
    if isinstance(has_part, dict):
        steps = [has_part]
    elif isinstance(has_part, list):
        steps = has_part
    else:
        steps = [dtreg_data]
    for step in steps:
        if not isinstance(step, dict):
            continue
        info = parse_single_step(step, filename)
        if info:
            results.append(info)
    return results


Helpers defined.


## 2. Generate TriG files for all records

Creates unsigned TriG files for every non-DONE record in `loom_records.txt`.
Files go into `outputs/<RESOURCE_ID>/`. Review them before publishing.


In [ ]:
# BATCH GENERATE — creates TriG files for all non-DONE records (does NOT publish)
import time

def parse_loom_records(path="../loom_records.txt"):
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"): continue
            parts = line.split()
            rid, stmts, repo = parts[0], int(parts[1]), parts[2]
            done = "DONE" in line
            records.append({"id": rid, "stmts": stmts, "repo": repo, "done": done})
    return records

all_records = parse_loom_records()
todo = [r for r in all_records if not r["done"]]
print(f"Records: {len(all_records)} total, {len(todo)} to process, {len(all_records)-len(todo)} DONE")

batch_results = {}  # rid -> {claims: [files], study: file, outcomes: [files]}
errors = []

for idx, rec in enumerate(todo):
    rid = rec["id"]
    repo = rec["repo"]
    print(f"\n{'='*60}")
    print(f"[{idx+1}/{len(todo)}] {rid} / {repo}")
    print(f"{'='*60}")
    
    rec_dir = Path(OUTPUT_DIR) / rid
    rec_dir.mkdir(parents=True, exist_ok=True)
    
    try:
        # 1. Fetch KL data
        _kl = fetch_json(f"{KL_API}/articles/get_article_by_id/?id={rid}")
        _article = _kl.get("article", _kl)
        _statements = _kl.get("statements", [])
        _datasets = _kl.get("basises", [])
        _kl_url = f"https://knowledgeloom.tib.eu/resource/{rid}"
        _source_doi = None
        for d in _datasets:
            if d.get("id", "").startswith("http"):
                _source_doi = d["id"]
                break
        print(f"  KL: {_article['name'][:60]}... ({len(_statements)} statements)")

        # 2. Fetch dtreg from GitLab
        _analyses = []
        _script_url = ""
        _input_data_urls = []
        _stmt_output_map = {}
        
        _tree = fetch_json(gitlab_api_url(repo, "repository/tree?per_page=100&ref=main"))
        _json_files = [f["name"] for f in _tree if f["name"].endswith(".json") and f["name"] != "metadata.json"]
        
        # Parse dtreg JSONs for study-level metadata
        for jf in _json_files:
            try:
                _ej = urllib.parse.quote(jf, safe="")
                _dtreg = fetch_json(gitlab_api_url(repo, f"repository/files/{_ej}/raw?ref=main"))
                _impl = get_prop(_dtreg, "is_implemented_by")
                if isinstance(_impl, str) and _impl.startswith("http"):
                    _script_url = _impl
                _infos = extract_analysis_steps(_dtreg, jf)
                _analyses.extend(_infos)
            except Exception:
                pass
        
        # Parse metadata.json for statement mapping + data URLs
        try:
            _em = urllib.parse.quote("utils/metadata.json", safe="")
            _metadata = fetch_json(gitlab_api_url(repo, f"repository/files/{_em}/raw?ref=main"))
            _fm = _metadata.get("file_mapping", {})
            _ms = _metadata.get("statements", {})
            
            # Input data URLs
            _data_exts = {"csv", "xlsx", "tsv", "rds", "rda", "dat", "sav", "txt"}
            for fname, finfo in _fm.items():
                ext = fname.rsplit(".", 1)[-1].lower() if "." in fname else ""
                if ext in _data_exts and finfo.get("resource_url"):
                    _input_data_urls.append((fname, finfo["resource_url"]))
            
            # Statement-to-output mapping
            _label_to_json = {}
            for _sid, _sinfo in _ms.items():
                _lbl = _sinfo.get("label", "")
                _jorig = _sinfo.get("json_file_name", "")
                _mapped = _fm.get(_jorig, {}).get("mapped_name", "")
                if _lbl and _mapped:
                    _label_to_json[_lbl] = _mapped
                    _label_to_json[_lbl.strip()] = _mapped
            
            for _lbl, _jfname in _label_to_json.items():
                try:
                    _ej = urllib.parse.quote(_jfname, safe="")
                    _dt = fetch_json(gitlab_api_url(repo, f"repository/files/{_ej}/raw?ref=main"))
                    _hp = get_prop(_dt, "has_part")
                    _steps = _hp if isinstance(_hp, list) else [_hp] if isinstance(_hp, dict) else []
                    for _step in reversed(_steps):
                        if isinstance(_step, dict):
                            _oi = extract_step_output(_step)
                            if _oi.output_values or _oi.output_label or _oi.viz_url:
                                _stmt_output_map[_lbl] = _oi
                                break
                except Exception:
                    pass
        except Exception:
            pass
        
        print(f"  dtreg: {len(_analyses)} steps, {len(_stmt_output_map)} outputs, {len(_input_data_urls)} data URLs")

        # 3. Generate claims
        _claim_files = []
        for i, stmt in enumerate(_statements):
            # Reuse create_claim logic with local vars
            ds = Dataset()
            bind_all(ds)
            this_np = make_head(ds)
            _stmt_id = stmt.get("statement_id", f"s{i}")
            _aida_sentence = stmt["label"]
            _aida_uri = URIRef(f"https://knowledgeloom.tib.eu/resource/{rid}/statement/{_stmt_id}")
            _type_info = stmt.get("type", {})
            _type_name = _type_info.get("name", "").lower().replace(" ", "_")
            _type_hash = _type_info.get("type_id", "")
            claim_id = slugify(f"kl-claim-{_stmt_id}")
            claim_uri = URIRef(str(TEMP_NP) + "/" + claim_id)
            a = ds.graph(URIRef(TEMP_NP + "/assertion"))
            a.add((claim_uri, RDF.type, SCIENCELIVE["FORRT-Claim"]))
            # Auto-map claim type
            if _type_name in ("group_comparison", "regression_analysis", "correlation_analysis"):
                a.add((claim_uri, RDF.type, SCIENCELIVE["statistical_significance-FORRT-Claim"]))
            elif _type_name == "algorithm_evaluation":
                a.add((claim_uri, RDF.type, SCIENCELIVE["model_performance-FORRT-Claim"]))
            elif _type_name in ("descriptive_statistics", "factor_analysis"):
                a.add((claim_uri, RDF.type, SCIENCELIVE["descriptive_pattern-FORRT-Claim"]))
            elif _type_name == "data_preprocessing":
                a.add((claim_uri, RDF.type, SCIENCELIVE["data_quality-FORRT-Claim"]))
            else:
                a.add((claim_uri, RDF.type, SCIENCELIVE["statistical_significance-FORRT-Claim"]))
            a.add((claim_uri, RDFS.label, Literal(_aida_sentence)))
            a.add((claim_uri, SCIENCELIVE["asAidaStatement"], _aida_uri))
            if _source_doi:
                a.add((claim_uri, DCT.source, URIRef(_source_doi)))
            for ai in _analyses:
                if ai.input_label:
                    desc = ai.input_label
                    if ai.input_rows and ai.input_cols: desc += f" ({ai.input_rows} rows x {ai.input_cols} columns)"
                    a.add((claim_uri, SCIENCELIVE["hasDataSource"], Literal(desc)))
                    break
            _clabel = f"FORRT Claim: {_aida_sentence[:80]}{'...' if len(_aida_sentence) > 80 else ''}"
            make_provenance(ds)
            make_pubinfo(ds, this_np, _clabel, FORRT_CLAIM_TEMPLATE, claim_uri)
            _cf = rec_dir / f"{rid}-claim-{i+1}.trig"
            ds.serialize(destination=str(_cf), format='trig')
            _claim_files.append(_cf)

        # 4. Generate study
        ds = Dataset()
        bind_all(ds)
        this_np = make_head(ds)
        _study_id = slugify(f"kl-study-{rid}")
        _study_uri = URIRef(str(TEMP_NP) + "/" + _study_id)
        a = ds.graph(URIRef(TEMP_NP + "/assertion"))
        a.add((_study_uri, RDF.type, SCIENCELIVE["FORRT-Replication-Study"]))
        a.add((_study_uri, RDF.type, SCIENCELIVE["Reproduction-Study"]))
        _slabel = f"Replication study: {_article.get('name', rid)}"
        a.add((_study_uri, RDFS.label, Literal(_slabel)))
        _abstract = _article.get("abstract", "")
        _scope = f"Reproduction of analyses from Knowledge Loom record: {_article.get('name', rid)}"
        if _abstract: _scope += f". {_abstract}"
        a.add((_study_uri, SCIENCELIVE["hasScopeDescription"], Literal(_scope)))
        _types = set(s.get("type", {}).get("name", "") for s in _statements if s.get("type", {}).get("name"))
        _method = f"Analysis types: {', '.join(sorted(_types))}." if _types else ""
        _method += " Machine-readable descriptions generated using dtreg and published in the TIB Knowledge Loom."
        a.add((_study_uri, SCIENCELIVE["hasMethodologyDescription"], Literal(_method.strip())))
        if _analyses:
            _methods, _packages, _runtimes, _input_descs, _input_urls = set(), set(), set(), set(), set()
            for ai in _analyses:
                if ai.method_call: _methods.add(ai.method_call.split(chr(10))[0].strip())
                elif ai.method_label and ai.package_name: _methods.add(f"{ai.package_name}::{ai.method_label}()")
                if ai.package_name: _packages.add(f"{ai.package_name} {ai.package_version}".strip())
                if ai.runtime_name: _runtimes.add(f"{ai.runtime_name} {ai.runtime_version}".strip())
                if ai.input_label:
                    desc = ai.input_label
                    if ai.input_rows and ai.input_cols: desc += f" ({ai.input_rows} rows x {ai.input_cols} columns)"
                    _input_descs.add(desc)
                if ai.input_source_url: _input_urls.add(ai.input_source_url)
            if _methods: a.add((_study_uri, SCIENCELIVE["executesMethod"], Literal("; ".join(sorted(_methods)))))
            if _packages: a.add((_study_uri, SCIENCELIVE["usesSoftwarePackage"], Literal("; ".join(sorted(_packages)))))
            if _runtimes: a.add((_study_uri, SCIENCELIVE["hasRuntimeEnvironment"], Literal("; ".join(sorted(_runtimes)))))
            if _input_descs: a.add((_study_uri, SCIENCELIVE["hasInputDataDescription"], Literal("; ".join(sorted(_input_descs)))))
            for url in sorted(_input_urls): a.add((_study_uri, SCIENCELIVE["hasInputDataSource"], URIRef(url)))
        for fname, url in _input_data_urls:
            a.add((_study_uri, SCIENCELIVE["hasInputDataset"], URIRef(url)))
        if _script_url: a.add((_study_uri, SCIENCELIVE["hasAnalysisScript"], URIRef(_script_url)))
        a.add((_study_uri, SCIENCELIVE["hasLoomRecord"], URIRef(_kl_url)))
        make_provenance(ds)
        make_pubinfo(ds, this_np, _slabel, FORRT_KL_STUDY_TEMPLATE, _study_uri)
        _sf = rec_dir / f"{rid}-study.trig"
        ds.serialize(destination=str(_sf), format='trig')

        # 5. Generate outcomes
        _outcome_files = []
        for i, stmt in enumerate(_statements):
            ds = Dataset()
            bind_all(ds)
            this_np = make_head(ds)
            _oid = slugify(f"kl-outcome-{rid}-{i+1}")
            _ouri = URIRef(str(TEMP_NP) + "/" + _oid)
            _slbl = stmt["label"]
            _tn = stmt.get("type", {}).get("name", "analysis")
            _olabel = f"Outcome ({_tn}): {_slbl[:60]}{'...' if len(_slbl) > 60 else ''}"
            _today = datetime.now(timezone.utc).strftime("%Y-%m-%d")
            a = ds.graph(URIRef(TEMP_NP + "/assertion"))
            a.add((_ouri, RDF.type, SCIENCELIVE["FORRT-Replication-Outcome"]))
            a.add((_ouri, RDFS.label, Literal(_olabel)))
            a.add((_ouri, SCIENCELIVE["hasOutcomeRepository"], URIRef(_kl_url)))
            a.add((_ouri, SCHEMA.endDate, Literal(_today, datatype=XSD.date)))
            a.add((_ouri, SCIENCELIVE["hasValidationStatus"], SCIENCELIVE["Validated"]))
            a.add((_ouri, SCIENCELIVE["hasConfidenceLevel"], SCIENCELIVE["HighConfidence"]))
            a.add((_ouri, SCIENCELIVE["hasConclusionDescription"], Literal(f"Knowledge Loom verified: {_slbl}")))
            _oi = _stmt_output_map.get(_slbl) or _stmt_output_map.get(_slbl.strip())
            if _oi:
                if _oi.target_variable:
                    a.add((_ouri, SCIENCELIVE["hasTargetVariable"], Literal(_oi.target_variable)))
                if _oi.output_label:
                    a.add((_ouri, SCIENCELIVE["hasOutputDescription"], Literal(_oi.output_label)))
                if _oi.output_values:
                    vals_str = "; ".join(f"{k} = {v}" for k, v in _oi.output_values.items())
                    a.add((_ouri, SCIENCELIVE["hasResultValues"], Literal(vals_str)))
                if _oi.output_columns:
                    a.add((_ouri, SCIENCELIVE["hasResultMetrics"], Literal(", ".join(_oi.output_columns))))
                if _oi.viz_url:
                    a.add((_ouri, SCIENCELIVE["hasResultVisualization"], URIRef(_oi.viz_url)))
                if _oi.step_label:
                    a.add((_ouri, SCIENCELIVE["hasAnalysisDescription"], Literal(_oi.step_label)))
                _evidence = f"Reproduced {_tn}: {_oi.step_label or _slbl}."
                if _oi.output_values:
                    _tv = "; ".join(f"{k}={v}" for k, v in list(_oi.output_values.items())[:5])
                    _evidence += f" Result: {_tv}."
                if _oi.input_label:
                    a.add((_ouri, SCIENCELIVE["hasInputDataDescription"], Literal(_oi.input_label)))
                if _oi.input_source_url:
                    a.add((_ouri, SCIENCELIVE["hasInputDataSource"], URIRef(_oi.input_source_url)))
            else:
                _evidence = f"Machine-readable analysis proof ({_tn}) published in the TIB Knowledge Loom."
            a.add((_ouri, SCIENCELIVE["hasEvidenceDescription"], Literal(_evidence)))
            a.add((_ouri, SCIENCELIVE["hasMachineReadableProof"], URIRef(_kl_url)))
            _th = stmt.get("type", {}).get("type_id", "")
            if _th in DTREG_TYPE_MAP:
                a.add((_ouri, SCIENCELIVE["hasAnalysisType"], DTREG_TYPE_MAP[_th]))
            make_provenance(ds)
            make_pubinfo(ds, this_np, _olabel, FORRT_KL_OUTCOME_TEMPLATE, _ouri)
            _of = rec_dir / f"{rid}-outcome-{i+1}.trig"
            ds.serialize(destination=str(_of), format='trig')
            _outcome_files.append(_of)

        _total = len(_claim_files) + 1 + len(_outcome_files)
        batch_results[rid] = {"claims": _claim_files, "study": _sf, "outcomes": _outcome_files}
        print(f"  Generated {_total} TriG files ({len(_claim_files)} claims + 1 study + {len(_outcome_files)} outcomes)")

    except Exception as e:
        errors.append((rid, str(e)))
        print(f"  ERROR: {e}")
    
    time.sleep(0.5)  # be nice to APIs

print(f"\n{'='*60}")
print(f"BATCH COMPLETE: {len(batch_results)}/{len(todo)} records generated")
if errors:
    print(f"Errors ({len(errors)}):")
    for rid, err in errors:
        print(f"  {rid}: {err}")
total_files = sum(len(r['claims']) + 1 + len(r['outcomes']) for r in batch_results.values())
print(f"Total TriG files: {total_files}")
print(f"\nReview files in {OUTPUT_DIR}<RESOURCE_ID>/ before publishing.")


## 3. Publish

After reviewing the generated TriG files, run the cell below.
For each record, publishes in correct order:
1. **Claims** → collects resource URIs
2. **Study** → re-generated with `targetsClaim` links → collects study URI
3. **Outcomes** → re-generated with `isOutcomeOf` link

**This is irreversible** — nanopubs cannot be deleted from the network.


In [ ]:
# BATCH PUBLISH — sign and publish in correct order per record
# WARNING: Only run after reviewing the generated TriG files!

PUBLISH_BATCH = False  # Set to True to actually publish

if not PUBLISH_BATCH:
    print("Set PUBLISH_BATCH = True to publish. Review the TriG files first!")
else:
    published = {}  # rid -> {claim_uris, study_uri, outcome_uris}
    publish_errors = []
    
    for rid, files in batch_results.items():
        print(f"\n{'='*60}")
        print(f"Publishing {rid}...")
        rec_dir = Path(OUTPUT_DIR) / rid
        pub = {"claim_uris": [], "study_uri": None, "outcome_uris": []}
        
        try:
            # --- 1. Publish claims ---
            for cf in files["claims"]:
                _suffix = cf.stem.replace(f"{rid}-claim-", "kl-claim-")
                _np_uri, _res_uri = sign_and_publish(cf, resource_suffix=_suffix)
                pub["claim_uris"].append(_res_uri)
                print(f"  Claim: {_res_uri}")
            
            # --- 2. Re-generate study WITH targetsClaim, then publish ---
            # Re-fetch KL data for this record
            _kl = fetch_json(f"{KL_API}/articles/get_article_by_id/?id={rid}")
            _article = _kl.get("article", _kl)
            _statements = _kl.get("statements", [])
            _kl_url = f"https://knowledgeloom.tib.eu/resource/{rid}"
            
            # Look up GitLab repo from loom_records.txt
            _repo = ""
            with open("../loom_records.txt") as _f:
                for _line in _f:
                    if _line.strip().startswith(rid):
                        _repo = _line.split()[2]
                        break
            
            # Fetch dtreg analyses + metadata (reuse from batch_results context)
            _analyses_pub = []
            _script_url_pub = ""
            _input_data_urls_pub = []
            if _repo:
                _tree = fetch_json(gitlab_api_url(_repo, "repository/tree?per_page=100&ref=main"))
                _json_files = [f["name"] for f in _tree if f["name"].endswith(".json") and f["name"] != "metadata.json"]
                for jf in _json_files:
                    try:
                        _ej = urllib.parse.quote(jf, safe="")
                        _dtreg = fetch_json(gitlab_api_url(_repo, f"repository/files/{_ej}/raw?ref=main"))
                        _impl = get_prop(_dtreg, "is_implemented_by")
                        if isinstance(_impl, str) and _impl.startswith("http"): _script_url_pub = _impl
                        _analyses_pub.extend(extract_analysis_steps(_dtreg, jf))
                    except: pass
                try:
                    _em = urllib.parse.quote("utils/metadata.json", safe="")
                    _meta = fetch_json(gitlab_api_url(_repo, f"repository/files/{_em}/raw?ref=main"))
                    _fm = _meta.get("file_mapping", {})
                    _data_exts = {"csv", "xlsx", "tsv", "rds", "rda", "dat", "sav", "txt"}
                    for fn, fi in _fm.items():
                        ext = fn.rsplit(".", 1)[-1].lower() if "." in fn else ""
                        if ext in _data_exts and fi.get("resource_url"):
                            _input_data_urls_pub.append((fn, fi["resource_url"]))
                except: pass
            
            # Build study with real claim URIs
            ds = Dataset()
            bind_all(ds)
            this_np = make_head(ds)
            _study_id = slugify(f"kl-study-{rid}")
            _study_uri_local = URIRef(str(TEMP_NP) + "/" + _study_id)
            a = ds.graph(URIRef(TEMP_NP + "/assertion"))
            a.add((_study_uri_local, RDF.type, SCIENCELIVE["FORRT-Replication-Study"]))
            a.add((_study_uri_local, RDF.type, SCIENCELIVE["Reproduction-Study"]))
            _slabel = f"Replication study: {_article.get('name', rid)}"
            a.add((_study_uri_local, RDFS.label, Literal(_slabel)))
            # targetsClaim with REAL published URIs
            for _curi in pub["claim_uris"]:
                a.add((_study_uri_local, SCIENCELIVE["targetsClaim"], URIRef(_curi)))
            _abstract = _article.get("abstract", "")
            _scope = f"Reproduction of analyses from Knowledge Loom record: {_article.get('name', rid)}"
            if _abstract: _scope += f". {_abstract}"
            a.add((_study_uri_local, SCIENCELIVE["hasScopeDescription"], Literal(_scope)))
            _types = set(s.get("type", {}).get("name", "") for s in _statements if s.get("type", {}).get("name"))
            _method = f"Analysis types: {', '.join(sorted(_types))}." if _types else ""
            _method += " Machine-readable descriptions generated using dtreg and published in the TIB Knowledge Loom."
            a.add((_study_uri_local, SCIENCELIVE["hasMethodologyDescription"], Literal(_method.strip())))
            if _analyses_pub:
                _m, _p, _r, _id, _iu = set(), set(), set(), set(), set()
                for ai in _analyses_pub:
                    if ai.method_call: _m.add(ai.method_call.split(chr(10))[0].strip())
                    elif ai.method_label and ai.package_name: _m.add(f"{ai.package_name}::{ai.method_label}()")
                    if ai.package_name: _p.add(f"{ai.package_name} {ai.package_version}".strip())
                    if ai.runtime_name: _r.add(f"{ai.runtime_name} {ai.runtime_version}".strip())
                    if ai.input_label:
                        d = ai.input_label
                        if ai.input_rows and ai.input_cols: d += f" ({ai.input_rows} rows x {ai.input_cols} columns)"
                        _id.add(d)
                    if ai.input_source_url: _iu.add(ai.input_source_url)
                if _m: a.add((_study_uri_local, SCIENCELIVE["executesMethod"], Literal("; ".join(sorted(_m)))))
                if _p: a.add((_study_uri_local, SCIENCELIVE["usesSoftwarePackage"], Literal("; ".join(sorted(_p)))))
                if _r: a.add((_study_uri_local, SCIENCELIVE["hasRuntimeEnvironment"], Literal("; ".join(sorted(_r)))))
                if _id: a.add((_study_uri_local, SCIENCELIVE["hasInputDataDescription"], Literal("; ".join(sorted(_id)))))
                for url in sorted(_iu): a.add((_study_uri_local, SCIENCELIVE["hasInputDataSource"], URIRef(url)))
            for fn, url in _input_data_urls_pub:
                a.add((_study_uri_local, SCIENCELIVE["hasInputDataset"], URIRef(url)))
            if _script_url_pub: a.add((_study_uri_local, SCIENCELIVE["hasAnalysisScript"], URIRef(_script_url_pub)))
            a.add((_study_uri_local, SCIENCELIVE["hasLoomRecord"], URIRef(_kl_url)))
            make_provenance(ds)
            make_pubinfo(ds, this_np, _slabel, FORRT_KL_STUDY_TEMPLATE, _study_uri_local)
            _sf = rec_dir / f"{rid}-study.trig"
            ds.serialize(destination=str(_sf), format='trig')
            validate_trig(_sf)
            _np_uri, _study_res_uri = sign_and_publish(_sf, resource_suffix=_study_id)
            pub["study_uri"] = _study_res_uri
            print(f"  Study: {_study_res_uri} (links to {len(pub['claim_uris'])} claims)")
            
            # --- 3. Re-generate outcomes WITH isOutcomeOf real study URI, then publish ---
            # Re-build stmt_output_map for this record
            _stmt_output_map_pub = {}
            if _repo:
                try:
                    _em = urllib.parse.quote("utils/metadata.json", safe="")
                    _meta = fetch_json(gitlab_api_url(_repo, f"repository/files/{_em}/raw?ref=main"))
                    _ms = _meta.get("statements", {})
                    _fm = _meta.get("file_mapping", {})
                    _l2j = {}
                    for _sid, _si in _ms.items():
                        _lbl = _si.get("label", "")
                        _jo = _si.get("json_file_name", "")
                        _mp = _fm.get(_jo, {}).get("mapped_name", "")
                        if _lbl and _mp:
                            _l2j[_lbl] = _mp
                            _l2j[_lbl.strip()] = _mp
                    for _lbl, _jfn in _l2j.items():
                        try:
                            _ej = urllib.parse.quote(_jfn, safe="")
                            _dt = fetch_json(gitlab_api_url(_repo, f"repository/files/{_ej}/raw?ref=main"))
                            _hp = get_prop(_dt, "has_part")
                            _stp = _hp if isinstance(_hp, list) else [_hp] if isinstance(_hp, dict) else []
                            for _s in reversed(_stp):
                                if isinstance(_s, dict):
                                    _oi = extract_step_output(_s)
                                    if _oi.output_values or _oi.output_label or _oi.viz_url:
                                        _stmt_output_map_pub[_lbl] = _oi
                                        break
                        except: pass
                except: pass
            
            for i, stmt in enumerate(_statements):
                ds = Dataset()
                bind_all(ds)
                this_np = make_head(ds)
                _oid = slugify(f"kl-outcome-{rid}-{i+1}")
                _ouri = URIRef(str(TEMP_NP) + "/" + _oid)
                _slbl = stmt["label"]
                _tn = stmt.get("type", {}).get("name", "analysis")
                _olabel = f"Outcome ({_tn}): {_slbl[:60]}{'...' if len(_slbl) > 60 else ''}"
                _today = datetime.now(timezone.utc).strftime("%Y-%m-%d")
                a = ds.graph(URIRef(TEMP_NP + "/assertion"))
                a.add((_ouri, RDF.type, SCIENCELIVE["FORRT-Replication-Outcome"]))
                a.add((_ouri, RDFS.label, Literal(_olabel)))
                # REAL study URI
                a.add((_ouri, SCIENCELIVE["isOutcomeOf"], URIRef(_study_res_uri)))
                a.add((_ouri, SCIENCELIVE["hasOutcomeRepository"], URIRef(_kl_url)))
                a.add((_ouri, SCHEMA.endDate, Literal(_today, datatype=XSD.date)))
                a.add((_ouri, SCIENCELIVE["hasValidationStatus"], SCIENCELIVE["Validated"]))
                a.add((_ouri, SCIENCELIVE["hasConfidenceLevel"], SCIENCELIVE["HighConfidence"]))
                a.add((_ouri, SCIENCELIVE["hasConclusionDescription"], Literal(f"Knowledge Loom verified: {_slbl}")))
                _oi = _stmt_output_map_pub.get(_slbl) or _stmt_output_map_pub.get(_slbl.strip())
                if _oi:
                    if _oi.target_variable: a.add((_ouri, SCIENCELIVE["hasTargetVariable"], Literal(_oi.target_variable)))
                    if _oi.output_label: a.add((_ouri, SCIENCELIVE["hasOutputDescription"], Literal(_oi.output_label)))
                    if _oi.output_values:
                        vs = "; ".join(f"{k} = {v}" for k, v in _oi.output_values.items())
                        a.add((_ouri, SCIENCELIVE["hasResultValues"], Literal(vs)))
                    if _oi.output_columns: a.add((_ouri, SCIENCELIVE["hasResultMetrics"], Literal(", ".join(_oi.output_columns))))
                    if _oi.viz_url: a.add((_ouri, SCIENCELIVE["hasResultVisualization"], URIRef(_oi.viz_url)))
                    if _oi.step_label: a.add((_ouri, SCIENCELIVE["hasAnalysisDescription"], Literal(_oi.step_label)))
                    _ev = f"Reproduced {_tn}: {_oi.step_label or _slbl}."
                    if _oi.output_values:
                        _tv = "; ".join(f"{k}={v}" for k, v in list(_oi.output_values.items())[:5])
                        _ev += f" Result: {_tv}."
                    if _oi.input_label: a.add((_ouri, SCIENCELIVE["hasInputDataDescription"], Literal(_oi.input_label)))
                    if _oi.input_source_url: a.add((_ouri, SCIENCELIVE["hasInputDataSource"], URIRef(_oi.input_source_url)))
                else:
                    _ev = f"Machine-readable analysis proof ({_tn}) published in the TIB Knowledge Loom."
                a.add((_ouri, SCIENCELIVE["hasEvidenceDescription"], Literal(_ev)))
                a.add((_ouri, SCIENCELIVE["hasMachineReadableProof"], URIRef(_kl_url)))
                _th = stmt.get("type", {}).get("type_id", "")
                if _th in DTREG_TYPE_MAP: a.add((_ouri, SCIENCELIVE["hasAnalysisType"], DTREG_TYPE_MAP[_th]))
                make_provenance(ds)
                make_pubinfo(ds, this_np, _olabel, FORRT_KL_OUTCOME_TEMPLATE, _ouri)
                _of = rec_dir / f"{rid}-outcome-{i+1}.trig"
                ds.serialize(destination=str(_of), format='trig')
                validate_trig(_of)
                _np_uri, _ores_uri = sign_and_publish(_of, resource_suffix=_oid)
                pub["outcome_uris"].append(_ores_uri)
                print(f"  Outcome {i+1}: {_ores_uri}")
            
            published[rid] = pub
            print(f"  DONE: {len(pub['claim_uris'])} claims + 1 study + {len(pub['outcome_uris'])} outcomes")
            
        except Exception as e:
            publish_errors.append((rid, str(e)))
            print(f"  ERROR: {e}")
            import traceback; traceback.print_exc()
        
        time.sleep(1)  # pace publishing
    
    # Summary
    print(f"\n{'='*60}")
    print(f"PUBLISHED: {len(published)}/{len(batch_results)} records")
    total_nps = sum(len(p['claim_uris']) + 1 + len(p['outcome_uris']) for p in published.values())
    print(f"Total nanopublications: {total_nps}")
    if publish_errors:
        print(f"\nErrors ({len(publish_errors)}):")
        for rid, err in publish_errors:
            print(f"  {rid}: {err}")
    print(f"\nDon't forget to mark published records as DONE in loom_records.txt")
